## Imports

In [ ]:
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv
import os

## Configuration

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

## Tools

In [ ]:
# Funciones importadas desde otras partes del proyecto
# TODO realizar los imports correctamente cuando las herramientas esten listas
from tools import rag_tool_function, web_search_function

tools = [
    Tool(
        name="RAGTool",
        func=rag_tool_function,
        description="Usa esta herramienta para responder preguntas basadas en los apuntes del curso. Úsala por defecto."
    ),
    Tool(
        name="WebSearchTool",
        func=web_search_function,
        description="Usa esta herramienta si el usuario explícitamente pide buscar en internet o menciona fuentes externas."
    )
]

## Memory for Context

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

## Prompt Context

In [ ]:
agent_prompt = """
Eres un asistente conversacional experto en Inteligencia Artificial. 
Tu trabajo es ayudar a los estudiantes a responder preguntas basadas en sus apuntes del curso del primer semestre 2025. 

- Usa la herramienta RAG para buscar en los apuntes.
- Usa la herramienta de búsqueda en internet **solo si el usuario lo solicita explícitamente** con frases como "busca en internet", "verifica en la web", etc.
- Mantén el contexto de las preguntas anteriores para responder con coherencia.
- Sé claro, conciso y evita responder "no sé" si hay información en los apuntes.

Comienza ahora.
"""


## Init Agent

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125", temperature=0)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)


## Test

In [ ]:
pregunta = input("Usuario: ")
respuesta = ""

if "internet" in pregunta.lower():
    respuesta = agent.run("Usa WebSearchTool: " + pregunta)
else:
    respuesta = agent.run(pregunta)

print(f"Agente: {respuesta}")
